In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import missingno as msno
%matplotlib inline
import csv

from functools import reduce

# 1. compare based on doi

## import data

In [ ]:
# data from pubmed
pubmed_doi = pd.read_csv(f'../data/preprocessing_pubmed/concat_all_doi_pubmed.csv').drop('Unnamed: 0', axis=1)
# data from lens.org
lens_doi = pd.read_csv(f'../data/preprocessing_lens/concat_all_doi_lens.csv').drop(['Unnamed: 0'], axis=1)
# data from WOS 
wos_doi = pd.read_csv(f'../data/preprocessing_wos/concat_all_doi_wos.csv').drop(['Unnamed: 0'], axis=1)
# data from crossref
crossref_doi = pd.read_csv(f'../data/preprocessing_crossref/concat_all_doi_crossref.csv').drop(['Unnamed: 0','alternative-id'], axis=1)
# data from scopus 
scopus_doi = pd.read_csv(f'../data/preprocessing_scopus/concat_all_doi_scopus.csv').drop(['Unnamed: 0'], axis=1)

#### previsualize and look at the NaN values

In [ ]:
pubmed_doi.head()

In [ ]:
pubmed_doi.count().to_frame()

In [ ]:
##

In [ ]:
lens_doi.head()

In [ ]:
lens_doi.count().to_frame()

In [ ]:
#

In [ ]:
wos_doi.head()

In [ ]:
wos_doi.count().to_frame()

In [ ]:
crossref_doi.head()

In [ ]:
crossref_doi.count().to_frame()

In [ ]:
scopus_doi.head()

In [ ]:
scopus_doi.count().to_frame()

### PubMed studies

In [ ]:
## How many studies in PubMed ?
pubmed_doi.drop_duplicates(inplace = True)
print(">>>", " total number of PubMed publications : ", pubmed_doi.shape[0])

In [ ]:
## How many PubMed studies have DOI  ?
pubmed_have_doi_have_doi = pubmed_doi[pubmed_doi['DOI'].notnull()]
print(">>>", " total number of PubMed publications with DOI : ", pubmed_have_doi_have_doi.shape[0])

In [ ]:
## How many PubMed studies have DOI  ?
pubmed_have_doi_no_doi = pubmed_doi[pubmed_doi['DOI'].isnull()]
print(">>>", " total number of PubMed publications with no DOI : ", pubmed_have_doi_no_doi.shape[0])

In [ ]:
pubmed_have_doi_no_doi.count()

### WOS studies

In [ ]:
wos_doi.columns

In [ ]:
# wos_doi.drop_duplicates(subset=['UT (Unique WOS ID)'])

In [ ]:
## How many studies in WOS ?
wos_doi.drop_duplicates(inplace = True)
print(">>>", " total number of WOS publications : ", wos_doi.shape[0])

In [ ]:
## How many WOS studies have DOI  ?
wos_have_doi_have_doi = wos_doi[wos_doi['DOI'].notnull()]
print(">>>", " total number of WOS publications with DOI : ", wos_have_doi_have_doi.shape[0])

In [ ]:
## How many WOS studies have DOI  ?
wos_have_doi_no_doi = wos_doi[wos_doi['DOI'].isnull()]
print(">>>", " total number of WOS publications with no DOI : ", wos_have_doi_no_doi.shape[0])

In [ ]:
wos_have_doi_no_doi.count()

### lens studies

In [ ]:
## How many studies in lens ?
lens_doi.drop_duplicates(inplace = True)
print(">>>", " total number of lens publications : ", lens_doi.shape[0])

In [ ]:
## How many lens studies have DOI  ?
lens_have_doi_have_doi = lens_doi[lens_doi['DOI'].notnull()]
print(">>>", " total number of lens publications with DOI : ", lens_have_doi_have_doi.shape[0])

In [ ]:
## How many lens studies have DOI  ?
lens_have_doi_no_doi = lens_doi[lens_doi['DOI'].isnull()]
print(">>>", " total number of lens publications with no DOI : ", lens_have_doi_no_doi.shape[0])

In [ ]:
lens_have_doi_no_doi.count()

### crossref studies

In [ ]:
## How many studies in crossref ?
crossref_doi_drop_dup = crossref_doi.drop_duplicates(subset=['DOI'], keep='first') 
print(">>>", " total number of crossref publications : ", crossref_doi_drop_dup.shape[0])

In [ ]:
## How many crossref studies have DOI  ?
crossref_doi_drop_dup_have_doi = crossref_doi_drop_dup[crossref_doi_drop_dup['DOI'].notnull()]
print(">>>", " total number of crossref publications with DOI : ", crossref_doi_drop_dup_have_doi.shape[0])

In [ ]:
## How many crossref studies with no DOI  ?
crossref_doi_drop_dup_no_doi = crossref_doi_drop_dup[crossref_doi_drop_dup['DOI'].isnull()]
print(">>>", " total number of crossref publications with no DOI : ", crossref_doi_drop_dup_no_doi.shape[0])

## How many studies are sharing between databases all databases not scopus ?

In [ ]:
# compile the list of dataframes you want to merge
dfs = [pubmed_have_doi_have_doi, lens_have_doi_have_doi, wos_have_doi_have_doi, crossref_doi_drop_dup]
#
databases_merged = reduce(lambda  left,right: pd.merge(left,right,
                                                       on=['DOI'],
                                                       how='outer'), 
                          dfs)

In [ ]:
databases_merged

In [ ]:
databases_merged.count()

In [ ]:
databases_merged.drop_duplicates().count()

In [ ]:
databases_merged.reset_index()['DOI'].drop_duplicates()

### doi in scopus data and not in another

In [ ]:
scopus_doi_dropna_dup = scopus_doi[['DOI','EID']].dropna().applymap(lambda s:s.lower() if type(s) == str else s).drop_duplicates('DOI').set_index('DOI')
scopus_doi_dropna_dup

In [ ]:
other_databases_doi = databases_merged[['DOI']].applymap(lambda s:s.lower() if type(s) == str else s).drop_duplicates().reset_index().set_index('DOI')
other_databases_doi

In [ ]:
doi_data_concat = pd.concat([scopus_doi_dropna_dup,other_databases_doi], axis=1)
doi_data_concat

In [ ]:
scopus_doi_not_in_others_databases = doi_data_concat[doi_data_concat['index'].isna()].reset_index().drop(['EID','index'], axis= 1)
scopus_doi_not_in_others_databases

## get Unpaywall data via the API

In [ ]:
import os
myemail = 'your_email@example.com'

In [ ]:
import requests
import json
import time

def get_doi_data(doi):
    email = myemail or ''
    url = 'https://api.unpaywall.org/v2/{}?email={}'.format(doi,email)
    r = requests.get(url)
    data = r.json()
    response_time = r.elapsed.total_seconds()
    # simple rule for sleeping if responses are slow
    if response_time > 5.0:
        print(response_time, 'seconds for last request')
        print('Unpaywall slow to respond. Sleep for a few seconds.')
        time.sleep(int(response_time))
        print('Finished sleeping')
    return data

In [ ]:
list_scopus_doi = list(scopus_doi_not_in_others_databases['DOI'])

In [ ]:
list_scopus_doi

In [ ]:
len(list_scopus_doi)

In [ ]:
# Python code to split a list
# into sublists of given length.
#list_doi_all = list(databases_merged.reset_index()['DOI'])#[:26]
list_scopus_doi_split = [list_scopus_doi[x:x+1000] for x in range(0, len(list_scopus_doi), 1000)]

In [ ]:
len(list_scopus_doi)

In [ ]:
len(list_scopus_doi_split)

In [ ]:
#list_doi_split[0]

In [ ]:
#list_doi_split[1]

In [ ]:
#list_doi_split[6]

In [ ]:
# list_doi_split[250]

In [ ]:
# data_upw = pd.DataFrame()
for elt in range(21, 25): #len(list_doi_split)
    data_upw = []
    for doi in list_scopus_doi_split[elt] :
        data_doi = get_doi_data(doi)
#         data_upw = pd.concat([data_upw,data_doi])
        data_upw.append(data_doi)
    
    data_upw = pd.json_normalize(data_upw)
    data_upw.to_csv(f'D:\RESEARCH IN AFRICA\Manuscripts\Open access paper\data\scopus_doi_notin_other_and_upw_data_{elt}.csv')
    print(f"#+++>>> total number of publication found in unpaywall for list_scopus_doi_split_{elt} :", data_upw.shape[0]," #") 

In [ ]:
print(f"#+++>>> total number of publication found in unpaywall for list_doi_{elt} :", data_upw.shape[0]," #") 

In [ ]:
data_doi

In [ ]:
data_upw

In [ ]:
pd.json_normalize(data_upw)